## This file creates Key Stage 1 (KS1) scores based on month of birth

In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
os.chdir(r"C:\Users\uctvjla\OneDrive - University College London\Desktop\ECHILD_Synthetic")

In [4]:
# Check if the files '2_spine_jupyter.csv' and '1_people_jupyter.csv' exist
if not os.path.exists("2_spine_jupyter.csv") and not os.path.exists("1_people_jupyter.csv"):
    # If the files don't exist, source "2_spine.R" (assuming it contains necessary functions)
    # Here, you might need to replace 'source("2_spine.R")' with the equivalent Python code.
    # This might involve loading functions from another Python script or module.

    # Placeholder for the missing source function
    print("Source 2_spine_jupyter.ipynb equivalent Python code not provided.")
else:
    # If the files exist, read '1_people.csv' into a DataFrame
    people = pd.read_csv("1_people_jupyter.csv")

    # Create new columns in the DataFrame
    people['KS1_AGE_START'] = np.where(people['month_of_birth'] == "sep", 6, 5)
    people['KS1_ACADYR'] = "2010/2011"
    people['KS1_YEAROFBIRTH'] = np.where(people['month_of_birth'].isin(["sep", "oct", "nov", "dec"]), 2004, 2005)
    
    # Mapping month_of_birth to numeric values
    month_mapping = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
    people['KS1_MONTHOFBIRTH'] = people['month_of_birth'].map(month_mapping)
    
    # Creating KS1_MONTH_PART
    month_part_mapping = {"sep": 12, "oct": 11, "nov": 10, "dec": 9, "jan": 8, "feb": 7, "mar": 6, "apr": 5, "may": 4, "jun": 3, "jul": 2, "aug": 1}
    people['KS1_MONTH_PART'] = people['month_of_birth'].map(month_part_mapping)
    
    people['KS1_GENDER'] = np.where(people['gender'] == 0, "M", "F")

    gestational_age_order = ["less_than_28", "28_29", "30_31", "32", "33", "34", "35", "36", "37", "38", "39", "40", "41", "42"]
    people['gestational_age'] = pd.Categorical(people['gestational_age'], categories=gestational_age_order, ordered=True)
    
    people['gestational_age_num'] = people['gestational_age'].cat.codes + 1
    people['gestational_age_num_randomness'] = np.random.uniform(0.5, 1.5, size=len(people))

    increments = (1 - 5) / 12
    increments_randomness = np.random.uniform(0.7, 1.2, size=len(people))
    
    people['score_temp'] = np.where((people['gestational_age_num'] >= 1) & (people['gestational_age_num'] <= 12),
                                    5 - (people['gestational_age_num'] * increments * people['gestational_age_num_randomness']),
                                    np.nan)
    people['score_temp'] = np.where(people['gestational_age_num'] > 12,
                                    5 - ((people['gestational_age_num'] - ((people['gestational_age_num'] - 12) * 2)) * increments * people['gestational_age_num_randomness']),
                                    people['score_temp'])
    people['score_adjustment_gender'] = np.random.uniform(0, 0.1, size=len(people))
    people['score_adjustment_gender'] = np.where(people['gender'] == 0, 0, people['score_adjustment_gender'])

    people['score_adjustment_imd'] = np.random.uniform(0, 0.01, size=len(people))
    people['score_adjustment_imd'] = people['imd_deciles'] * people['score_adjustment_imd']

    people['score_adjustment_month_of_birth'] = np.random.uniform(0, 0.01, size=len(people))
    people['score_adjustment_month_of_birth'] = people['score_adjustment_month_of_birth'] * people['KS1_MONTH_PART']

    people['score_temp'] = people['score_temp'] + (people['score_temp'] * people['score_adjustment_gender']) + \
                           (people['score_temp'] * people['score_adjustment_imd']) + \
                           (people['score_temp'] * people['score_adjustment_month_of_birth'])

    # Normalising Score Range
    min_normalized_score = 85 #adjust as needed
    max_normalized_score = 115 #adjust as needed
    people['score_normalised'] = ((people['score_temp'] - people['score_temp'].min()) / (people['score_temp'].max() - people['score_temp'].min())) * (max_normalized_score - min_normalized_score) + min_normalized_score

    rands = np.random.uniform(-0.1, 0.1, size=len(people))
    people['score_normalised'] = people['score_normalised'] + rands
    people['KS1_MATH'] = people['score_normalised']

    for i in people['gestational_age_num'].unique():
        coinflip = np.random.choice([0, 1], 1)[0]

        if coinflip == 0:
            random_val = np.random.uniform(-1, 1)
            people.loc[people['gestational_age_num'] == i, 'KS1_MATH'] = people.loc[people['gestational_age_num'] == i, 'KS1_MATH'] + random_val

    people['KS1_MATH'] = (people['KS1_MATH'] - people['KS1_MATH'].min()) / (people['KS1_MATH'].max() - people['KS1_MATH'].min()) * 100

    KS1_DATA = people[["PupilMatchingRefAnonymous", "KS1_ACADYR", "KS1_YEAROFBIRTH", "KS1_MONTHOFBIRTH", "KS1_AGE_START", "KS1_MONTH_PART", "KS1_GENDER", "KS1_MATH"]]
    KS1_DATA.to_csv("3_KS1_jupyter.csv", index=False)